### Timing: Depending on the Padlock number expected to generate, 5 min - 30 min

Using previous generated RO probes and bridges, this notebook is designed to assemble complete padlock. The key logic is to choose five ROs as the five colors for one cycle. And then choose bridge to constitute whole padlock, with a target that the two ends of the padlock are exposed under hybridization condition, and a total minimum free energy larger than a threshold.

environment: openFISH_probe  
The main function of this notebook is padlock join. One padlock assemble example is given. And it can be iterated to generate multiple padlocks for multiple cycles.


In [34]:
from nupack import *
import random
import itertools
from tqdm import tqdm
import pandas as pd
import numpy as np

class PadJoin():
    
    def __init__(self, RO1, RO2, Bridge_Database):
        
        ALL_POSSIBLE_CONNECTION = {
            "C__C": ["ta", "at", "tt", "aa"],
            "C__T": ["ta", "aa", "at", "tt"],
            "C__A": ["at", "tt", "ta", "aa"],
            "C__G": ['ta', 'at', 'tt', 'aa'],
            "T__T": ["aa", "at", "ta", "tt"],
            "T__C": ["at", "aa", "ta", "tt"],
            "T__A": ["at", "ta", "tt", "aa"],
            "T__G": ['at', 'aa', 'ta', 'tt'],
            "A__A": ["tt", "at", "ta", "aa"],
            "A__C": ["ta", "tt", "at", "aa"],
            "A__T": ["ta", "at", "tt", "aa"],
             'A__G': ['ta', 'tt', 'at', 'aa'],
             'G__G': ['ta', 'at', 'tt', 'aa'],
             'G__T': ["ta", "aa", "at", "tt"],
             'G__A': ["at", "tt", "ta", "aa"],
             'G__C': ['ta', 'at', 'tt', 'aa'],
        }
        
        self.Bridge_DataBase = Bridge_Database
        
        self.TEST_ORDER = [
            RO1 + ALL_POSSIBLE_CONNECTION[RO1[-1]+"__"+RO2[0]][0] + RO2,
            RO1 + ALL_POSSIBLE_CONNECTION[RO1[-1]+"__"+RO2[0]][1] + RO2,
            RO2 + ALL_POSSIBLE_CONNECTION[RO2[-1]+"__"+RO1[0]][0] + RO1,
            RO2 + ALL_POSSIBLE_CONNECTION[RO2[-1]+"__"+RO1[0]][1] + RO1,
            RO1 + ALL_POSSIBLE_CONNECTION[RO1[-1]+"__"+RO2[0]][2] + RO2,
            RO1 + ALL_POSSIBLE_CONNECTION[RO1[-1]+"__"+RO2[0]][3] + RO2,
            RO2 + ALL_POSSIBLE_CONNECTION[RO2[-1]+"__"+RO1[0]][2] + RO1,
            RO2 + ALL_POSSIBLE_CONNECTION[RO2[-1]+"__"+RO1[0]][3] + RO1,
        ]
        
        self.my_model = Model(material='dna', celsius=37, sodium = 0.075, magnesium=0.01) # Pad杂交条件
        
        self.p1_RNA = "AGCGACGGCTTCGGTAGCGT" # P1 = bridge[0:14] + p1_RNA
        self.p2_RNA = "TAGTCGCAGGTCCTCAGGCA" #P2 = p2_RNA + bridge[-14:]
        self.mRNA = "ACGCTACCGAAGCCGTCGCTGCTGCCTGAGGACCTGCGACTA"
        
    def reverse(self, seq):
        trans_dict = {"A":"T", "T":"A", "C":"G", "G":"C"}
        new_seq = []
        for ch in seq:
            new_seq.append(trans_dict[ch])
        return "".join(new_seq)[::-1]

    def pad_secondary(self, pad, dg_thred):
        
        Pad_strand = Strand(pad, name='padlock')
        t1 = Tube(strands={Pad_strand:1e-6}, complexes=SetSpec(max_size=1), name='Tube t1')
        tube_result = tube_analysis(tubes=[t1], compute=['pairs', 'mfe'], model=self.my_model)
        dg = tube_result['(padlock)'].mfe[-1][-2]
        
        if dg >= dg_thred:
            return "PASS"
        else:
            return "FAILED"
        
    def complex_secondary(self, pad, bridge):
        
        Pad_strand = Strand(pad, name='padlock')
        P1_strand = Strand(bridge[0:14] + "ta" + self.p1_RNA, name='p1_part')
        P2_strand = Strand(self.p2_RNA + "ta" + bridge[-14:], name='p2_part')
        mRNA_strand = Strand(self.mRNA, name = "mRNA")
        
        t1 = Tube(strands={Pad_strand:1e-6, P1_strand:1e-6, P2_strand:1e-6, mRNA_strand:1e-6}, complexes=SetSpec(max_size=4), name='Tube t1')
        tube_result = tube_analysis(tubes=[t1], compute=['pairs', 'mfe'], model=self.my_model)
        # print(tube_result)
        try:
            Dot_parens_plus_notation = str(tube_result["(mRNA+p2_part+padlock+p1_part)"].mfe[0][0])
        except KeyError:
            return "FAILED"
        
        if Dot_parens_plus_notation[153:160] == "......." and Dot_parens_plus_notation[80:87] == ".......":
            return "PASS"
        else:
            return "FAILED"
        
    def get_whole_pad(self, dg_thred = -1):

        Final_check = "FAIL"
        while Final_check == "FAIL":
            random_bridge = random.choice(self.Bridge_DataBase)
            reverse_random_bridge  = self.reverse(random_bridge)
            for partA in self.TEST_ORDER:
                whole_pad = reverse_random_bridge[-7:] + partA +reverse_random_bridge[0:23]
                # if self.kmer_check(whole_pad) == "FAILED":
                #     continue
                if self.pad_secondary(whole_pad, dg_thred) == "FAILED":
                    continue
                if self.complex_secondary(whole_pad, random_bridge) == "FAILED":
                    continue
                Final_check = "PASS"
                break

        return random_bridge, whole_pad
                    

## Choose five ROs for one cycle

Each cycle can label 10 genes with 5 ROs different combination. So, first choose five ROs, and then generate whole padlocks for each combination

If you already have chosen ROs, this can be skipped

In [21]:
candidate_RO_name = []
candidate_RO_seq = []

with open("RO_Pad_Design/Inclusion_RO.fa", "r") as handle:
    lines = handle.readlines()
    for i in tqdm(range(0, len(lines), 2)):
        line1 = lines[i].strip()[1:]
        line2 = lines[i+1].strip()
        candidate_RO_name.append(line1)
        candidate_RO_seq.append(line2)

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 72/72 [00:00<00:00, 433893.52it/s]


In [22]:
RO_idx = random.sample(list(np.arange(len(candidate_RO_seq))), 5)
combs = np.array(list(itertools.combinations([0,1,2,3,4], 2)))

ROs = np.array(candidate_RO_seq)[RO_idx]
RO_names = np.array(candidate_RO_name)[RO_idx]

for i, (i1,i2) in enumerate(combs):
    print(f"Comb{i+1}: {RO_names[i1]}: {ROs[i1]}  {RO_names[i2]}: {ROs[i2]}")

Comb1: RO_Test_20533: GTGCCAACTCGTGCGTACTC  RO_Test_25907: AACCGCTTACAACCGCTTAG
Comb2: RO_Test_20533: GTGCCAACTCGTGCGTACTC  RO_Test_23007: CGCTGAACTACGCTAAACTA
Comb3: RO_Test_20533: GTGCCAACTCGTGCGTACTC  RO_Test_19233: TTGACCGTACTTGACCGCAC
Comb4: RO_Test_20533: GTGCCAACTCGTGCGTACTC  RO_Test_4917: GGACTGATTCGGACTAATTC
Comb5: RO_Test_25907: AACCGCTTACAACCGCTTAG  RO_Test_23007: CGCTGAACTACGCTAAACTA
Comb6: RO_Test_25907: AACCGCTTACAACCGCTTAG  RO_Test_19233: TTGACCGTACTTGACCGCAC
Comb7: RO_Test_25907: AACCGCTTACAACCGCTTAG  RO_Test_4917: GGACTGATTCGGACTAATTC
Comb8: RO_Test_23007: CGCTGAACTACGCTAAACTA  RO_Test_19233: TTGACCGTACTTGACCGCAC
Comb9: RO_Test_23007: CGCTGAACTACGCTAAACTA  RO_Test_4917: GGACTGATTCGGACTAATTC
Comb10: RO_Test_19233: TTGACCGTACTTGACCGCAC  RO_Test_4917: GGACTGATTCGGACTAATTC


## Assemble one padlock when its two ROs are known

In [49]:
candidate_bridge_name = []
candidate_bridge_seq = []

with open("RO_Pad_Design/Inclusion_bridge.fa", "r") as handle:
    lines = handle.readlines()
    for i in tqdm(range(0, len(lines), 2)):
        line1 = lines[i].strip()[1:]
        line2 = lines[i+1].strip()
        candidate_bridge_name.append(line1)
        candidate_bridge_seq.append(line2)

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 34/34 [00:00<00:00, 283511.60it/s]


In [31]:
RO1 = "GTGCCAACTCGTGCGTACTC" # say AF488
RO2 = "AACCGCTTACAACCGCTTAG" # say AF546

In [36]:
PJ = PadJoin(RO1, RO2, candidate_bridge_seq)

In [40]:
# if this step takes too long, try lower the dg_thred, recommand -1, -3, -5
selected_bridge, joined_padlock = PJ.get_whole_pad(dg_thred = -3)
bridge_idx = candidate_bridge_seq.index(selected_bridge)
selected_bridge_name = candidate_bridge_name[bridge_idx]
print(f"Chosen bridge:  {selected_bridge}({selected_bridge_name})\nJoined padlock: {joined_padlock}")

Chosen bridge:  CTATAATTAGCGGACGCCGAAATGAGGGAG(bridge_Test_998)
Joined padlock: ATTATAGAACCGCTTACAACCGCTTAGtaGTGCCAACTCGTGCGTACTCCTCCCTCATTTCGGCGTCCGCTA


In [ ]:
# You can remove the selected bridge, and then generate another padlock
candidate_bridge_seq.remove(selected_bridge)
candidate_bridge_name.remove(selected_bridge_name)

## Assemble 10 padlocks for one cycle of chosen five ROs  

ROs have already be chosen in "Choose five ROs for one cycle" section, if not, provide RO sequence and RO names

In [52]:
# ROs = ['GTGCCAACTCGTGCGTACTC', 'AACCGCTTACAACCGCTTAG', 'CGCTGAACTACGCTAAACTA', 'TTGACCGTACTTGACCGCAC', 'GGACTGATTCGGACTAATTC']
# RO_names = ['RO_Test_20533', 'RO_Test_25907', 'RO_Test_23007', 'RO_Test_19233', 'RO_Test_4917']

candidate_bridge_name = []
candidate_bridge_seq = []

with open("RO_Pad_Design/Inclusion_bridge.fa", "r") as handle:
    lines = handle.readlines()
    for i in tqdm(range(0, len(lines), 2)):
        line1 = lines[i].strip()[1:]
        line2 = lines[i+1].strip()
        candidate_bridge_name.append(line1)
        candidate_bridge_seq.append(line2)

Col1 = []
Col2 = []
Col3 = []

combs = np.array(list(itertools.combinations([0,1,2,3,4], 2)))

for i, (i1,i2) in enumerate(combs):
    RO1 = ROs[i1]
    RO2 = ROs[i2]
    PJ = PadJoin(RO1, RO2, candidate_bridge_seq)
    selected_bridge, joined_padlock = PJ.get_whole_pad(dg_thred = -5)
    bridge_idx = candidate_bridge_seq.index(selected_bridge)
    selected_bridge_name = candidate_bridge_name[bridge_idx]
    
    print(f"Comb{i+1}: {RO_names[i1]}: {ROs[i1]}  {RO_names[i2]}: {ROs[i2]}")
    print(f"Chosen bridge:  {selected_bridge}({selected_bridge_name})\nJoined padlock: {joined_padlock}")
    print("------------------------------------------------------------------------------------------------------------------")

    Col1.append(joined_padlock)
    Col2.append(RO1)
    Col3.append(RO2)

    candidate_bridge_seq.remove(selected_bridge)
    candidate_bridge_name.remove(selected_bridge_name)

df = pd.DataFrame({
    'PadLock': Col1,
    'RO1': Col2,
    'RO2': Col3
})

# Results will be also saved
df.to_csv("RO_Pad_Design/Chosen_Pad_CycleTest.csv")

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 34/34 [00:00<00:00, 274242.95it/s]


Comb1: RO_Test_20533: GTGCCAACTCGTGCGTACTC  RO_Test_25907: AACCGCTTACAACCGCTTAG
Chosen bridge:  GATATTAGTACTAGCGCAGTGGTAGTCGCC(bridge_Test_93)
Joined padlock: TAATATCAACCGCTTACAACCGCTTAGtaGTGCCAACTCGTGCGTACTCGGCGACTACCACTGCGCTAGTAC
------------------------------------------------------------------------------------------------------------------
Comb2: RO_Test_20533: GTGCCAACTCGTGCGTACTC  RO_Test_23007: CGCTGAACTACGCTAAACTA
Chosen bridge:  TAAAGTATTCTCGACCCGGGTACACTTGAC(bridge_Test_1070)
Joined padlock: TACTTTACGCTGAACTACGCTAAACTAtaGTGCCAACTCGTGCGTACTCGTCAAGTGTACCCGGGTCGAGAA
------------------------------------------------------------------------------------------------------------------
Comb3: RO_Test_20533: GTGCCAACTCGTGCGTACTC  RO_Test_19233: TTGACCGTACTTGACCGCAC
Chosen bridge:  GATAGAAACGATAACGTTACGAGGCGATGA(bridge_Test_610)
Joined padlock: TTCTATCGTGCCAACTCGTGCGTACTCtaTTGACCGTACTTGACCGCACTCATCGCCTCGTAACGTTATCGT
-----------------------------------------------------------------------

## Assemble 10*n padlocks for n cycles

In [54]:
# The name of each cycle you want to design padlocks
Cycles = ['R12', 'R13']

In [55]:
candidate_RO_name = []
candidate_RO_seq = []

with open("RO_Pad_Design/Inclusion_RO.fa", "r") as handle:
    lines = handle.readlines()
    for i in tqdm(range(0, len(lines), 2)):
        line1 = lines[i].strip()[1:]
        line2 = lines[i+1].strip()
        candidate_RO_name.append(line1)
        candidate_RO_seq.append(line2)

candidate_bridge_name = []
candidate_bridge_seq = []

with open("RO_Pad_Design/Inclusion_bridge.fa", "r") as handle:
    lines = handle.readlines()
    for i in tqdm(range(0, len(lines), 2)):
        line1 = lines[i].strip()[1:]
        line2 = lines[i+1].strip()
        candidate_bridge_name.append(line1)
        candidate_bridge_seq.append(line2)

Col1 = []
Col2 = []
Col3 = []
Col4 = []

for cycle in tqdm(Cycles):
    RO_idx = random.sample(list(np.arange(len(candidate_RO_seq))), 5)
    combs = np.array(list(itertools.combinations([0,1,2,3,4], 2)))
    
    ROs = np.array(candidate_RO_seq)[RO_idx]
    RO_names = np.array(candidate_RO_name)[RO_idx]

    for r in ROs:
        candidate_RO_seq.remove(r)
    for r in RO_names:
        candidate_RO_name.remove(r)
    
    for i, (i1,i2) in enumerate(combs):
        RO1 = ROs[i1]
        RO2 = ROs[i2]
        PJ = PadJoin(RO1, RO2, candidate_bridge_seq)
        selected_bridge, joined_padlock = PJ.get_whole_pad(dg_thred = -5)
        bridge_idx = candidate_bridge_seq.index(selected_bridge)
        selected_bridge_name = candidate_bridge_name[bridge_idx]
        
        print(f"Cycle {cycle}, Comb{i+1}: {RO_names[i1]}: {ROs[i1]}  {RO_names[i2]}: {ROs[i2]}")
        print(f"Chosen bridge:  {selected_bridge}({selected_bridge_name})\nJoined padlock: {joined_padlock}")
        print("------------------------------------------------------------------------------------------------------------------")
    
        Col1.append(joined_padlock)
        Col2.append(RO1)
        Col3.append(RO2)
        Col4.append(cycle)
    
        candidate_bridge_seq.remove(selected_bridge)
        candidate_bridge_name.remove(selected_bridge_name)
    
df = pd.DataFrame({
    'PadLock': Col1,
    'RO1': Col2,
    'RO2': Col3,
    'Cycle': Col4
})
    
# Results will be also saved
df.to_csv("RO_Pad_Design/Padlock_CycleTest.csv") 

  0%|                                                                                                                                                                                                                             | 0/2 [00:00<?, ?it/s]

Cycle R12, Comb1: RO_Test_4814: GCAGGTATTCGCAGGCATTC  RO_Test_4733: GATTCGCCTTAATTCGCCTC
Chosen bridge:  ATTCTCTCGGTTATTAAGCTCCCGCGGATT(bridge_Test_223)
Joined padlock: AGAGAATGATTCGCCTTAATTCGCCTCtaGCAGGTATTCGCAGGCATTCAATCCGCGGGAGCTTAATAACCG
------------------------------------------------------------------------------------------------------------------
Cycle R12, Comb2: RO_Test_4814: GCAGGTATTCGCAGGCATTC  RO_Test_4416: CATAATTCCGCACTATTCCG
Chosen bridge:  ATGTAGCGATAATACGTTTGCGATCTGCAC(bridge_Test_166)
Joined padlock: GCTACATCATAATTCCGCACTATTCCGatGCAGGTATTCGCAGGCATTCGTGCAGATCGCAAACGTATTATC
------------------------------------------------------------------------------------------------------------------
Cycle R12, Comb3: RO_Test_4814: GCAGGTATTCGCAGGCATTC  RO_Test_10939: TATGAATCGCTATAAATCGC
Chosen bridge:  CAGACGATATGTTACCACGCGGTATGTCTG(bridge_Test_92)
Joined padlock: TCGTCTGTATGAATCGCTATAAATCGCtaGCAGGTATTCGCAGGCATTCCAGACATACCGCGTGGTAACATA
--------------------------------------------

 50%|██████████████████████████████████████████████████████████████████████████████████████████████████████████▌                                                                                                          | 1/2 [00:33<00:33, 33.05s/it]

Cycle R12, Comb10: RO_Test_10939: TATGAATCGCTATAAATCGC  RO_Test_11202: CTCGTATCGGCTCGCATCGG
Chosen bridge:  GCCTTAGATAAGTATGCGGTTAGCTAGGCC(bridge_Test_978)
Joined padlock: CTAAGGCTATGAATCGCTATAAATCGCtaCTCGTATCGGCTCGCATCGGGGCCTAGCTAACCGCATACTTAT
------------------------------------------------------------------------------------------------------------------
Cycle R13, Comb1: RO_Test_7403: CGTCCATTAGCGTCGATTAG  RO_Test_4727: TATTCGCTACCATTCGCTAC
Chosen bridge:  GATAGAAACGATAACGTTACGAGGCGATGA(bridge_Test_610)
Joined padlock: TTCTATCCGTCCATTAGCGTCGATTAGtaTATTCGCTACCATTCGCTACTCATCGCCTCGTAACGTTATCGT
------------------------------------------------------------------------------------------------------------------
Cycle R13, Comb2: RO_Test_7403: CGTCCATTAGCGTCGATTAG  RO_Test_17089: TACCGCTTGATACCGCTTAA
Chosen bridge:  GATCTAAATCTATGCGCCTAGACTGGACGC(bridge_Test_489)
Joined padlock: TTAGATCTACCGCTTGATACCGCTTAAatCGTCCATTAGCGTCGATTAGGCGTCCAGTCTAGGCGCATAGAT
----------------------------------------

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2/2 [01:12<00:00, 36.13s/it]

Cycle R13, Comb10: RO_Test_2826: GCACAATTTCGCAGTATTTC  RO_Test_15946: ATTATATCGGATCATATCGG
Chosen bridge:  TAATTCGAGAGACAATCGTCGAACGACACA(bridge_Test_567)
Joined padlock: CGAATTAATTATATCGGATCATATCGGaaGCACAATTTCGCAGTATTTCTGTGTCGTTCGACGATTGTCTCT
------------------------------------------------------------------------------------------------------------------
